# Сайт с базой данных — учебный проект

Разберём на примере **Автотеки ⭐** — простого сайта, где пользователи:

- регистрируются по **email + пароль**;
- **публикуют** PDF-отчёты автотеки Авито и получают **+100 ⭐**;
- **скачивают** чужие отчёты за **−10 ⭐** (повторно — бесплатно).

Стек: **FastAPI** (веб-сервер) + **SQLite** (база данных) + **SQLAlchemy** (ORM).

> Код приложения лежит в папке `app/`. Блокнот объясняет каждый слой и позволяет экспериментировать.

## Часть 1 — Зачем нужна база данных

Сайт хранит данные, которые должны пережить перезагрузку сервера:

| Сущность | Что храним |
|----------|------------|
| **Пользователь** | email, хеш пароля, баланс ⭐ |
| **Отчёт** | название, имя файла на диске, автор |
| **Скачивание** | кто и какой отчёт уже скачал (чтобы не списывать ⭐ дважды) |

```mermaid
erDiagram
    users ||--o{ reports : publishes
    users ||--o{ downloads : makes
    reports ||--o{ downloads : receives
    users {
        int id PK
        string email UK
        string password_hash
        int stars
    }
    reports {
        int id PK
        string title
        string filename
        int owner_id FK
    }
    downloads {
        int id PK
        int user_id FK
        int report_id FK
    }
```

**Файлы отчётов** хранятся на диске (`uploads/`), в БД — только метаданные. Так делают почти все сайты с загрузкой файлов.

## Часть 2 — SQLite и SQLAlchemy

**SQLite** — файловая БД (`autoteka.db`). Не нужен отдельный сервер — идеально для обучения.

**SQLAlchemy ORM** — Python-классы вместо ручного SQL. Смотрите [`app/database.py`](app/database.py) и [`app/models.py`](app/models.py).

In [1]:
import os
from pathlib import Path

# Перейдите в корень проекта (где лежит app/)
project_root = Path(r"c:\Users\user\Documents\py_projects\autoteka_stars")
os.chdir(project_root)
print("Рабочая папка:", project_root)

Рабочая папка: c:\Users\user\Documents\py_projects\autoteka_stars


In [2]:
from app.database import SessionLocal, init_db
from app.models import User, Report, Download

# Создаём таблицы (если ещё не созданы)
init_db()
print("Таблицы:", list(User.__table__.metadata.tables.keys()))

Таблицы: ['users', 'reports', 'downloads']


In [13]:
from sqlalchemy import select

db = SessionLocal()
users = db.scalars(select(User)).all()
print(f"Пользователей в БД: {len(users)}")
for u in users:
    print(f"  {u.id}: {u.email}, ⭐={u.stars}")
db.close()

Пользователей в БД: 5
  1: verify@test.com, ⭐=100
  2: alice@example.com, ⭐=100
  3: bob@example.com, ⭐=0
  4: notebook@test.com, ⭐=100
  5: msuya@ya.ru, ⭐=100


## Часть 3 — Авторизация

Пароль **никогда** не хранят в открытом виде. Используем **bcrypt** — односторонний хеш ([`app/auth.py`](app/auth.py)).

После входа в cookie-сессии сохраняется `user_id` (через `SessionMiddleware` в FastAPI).

In [15]:
from app import crud
from app.database import SessionLocal

db = SessionLocal()

# Регистрация тестового пользователя (пропустит, если email занят)
test_email = "demo@example.com"
try:
    user = crud.create_user(db, test_email, "secret123")
    print(f"Создан: {user.email}, ⭐={user.stars}")
except crud.CrudError as e:
    print(e)
    user = crud.get_user_by_email(db, test_email)

# Проверка пароля
ok = crud.authenticate_user(db, test_email, "secret123")
bad = crud.authenticate_user(db, test_email, "wrong")
print(f"Верный пароль: {ok is not None}")
print(f"Неверный пароль: {bad is not None}")
db.close()

Пользователь с таким email уже существует
Верный пароль: True
Неверный пароль: False


## Часть 4 — Бизнес-логика звёздочек

Вся логика ⭐ — в [`app/crud.py`](app/crud.py):

- `publish_report()` — сохраняет PDF в `uploads/`, создаёт запись в `reports`, **+100 ⭐**;
- `prepare_download()` — списывает **10 ⭐** при первом скачивании чужого отчёта.

In [ ]:
import io

from fastapi import UploadFile

from app import crud
from app.crud import PUBLISH_REWARD, DOWNLOAD_COST
from app.database import SessionLocal

db = SessionLocal()

alice = crud.get_user_by_email(db, "alice@example.com")
if alice is None:
    alice = crud.create_user(db, "alice@example.com", "pass1234")

bob = crud.get_user_by_email(db, "bob@example.com")
if bob is None:
    bob = crud.create_user(db, "bob@example.com", "pass1234")

sample_pdf = (project_root / "sample_report.pdf").read_bytes()

upload = UploadFile(filename="report.pdf", file=io.BytesIO(sample_pdf))
report = crud.publish_report(db, alice, "Toyota Camry 2018", upload)
db.refresh(alice)
print(f"Alice опубликовала отчёт #{report.id}, баланс: {alice.stars} ⭐ (ожидаем {PUBLISH_REWARD})")

db.refresh(bob)
print(f"Bob до скачивания: {bob.stars} ⭐")
_, path = crud.prepare_download(db, bob, report.id)
db.refresh(bob)
print(f"Bob скачал отчёт, баланс: {bob.stars} ⭐ (списано {DOWNLOAD_COST})")
print(f"Файл на диске: {path.name}")

# Повторное скачивание — бесплатно
stars_before = bob.stars
crud.prepare_download(db, bob, report.id)
db.refresh(bob)
print(f"Повторное скачивание: {bob.stars} ⭐ (без изменений: {bob.stars == stars_before})")

db.close()

## Часть 5 — FastAPI и HTML-страницы

FastAPI принимает HTTP-запросы и возвращает HTML (через Jinja2-шаблоны в `templates/`).

| Маршрут | Действие |
|---------|----------|
| `GET/POST /register` | Регистрация |
| `GET/POST /login` | Вход |
| `POST /logout` | Выход |
| `GET /` | Дашборд: баланс, список, загрузка |
| `GET /reports/{id}/download` | Скачивание файла |

### Запуск сервера

В терминале (из корня проекта):

```powershell
cd c:\Users\user\Documents\py_projects\autoteka_stars
.venv\Scripts\activate
uvicorn app.main:app --reload
```

Откройте в браузере: **http://127.0.0.1:8000**

In [ ]:
# Быстрая проверка, что приложение импортируется без ошибок
from app.main import app

routes = [r.path for r in app.routes if hasattr(r, "path")]
print("Маршруты:", sorted(set(routes)))

## Часть 6 — Проверка через HTTP из блокнота

Сначала **запустите сервер** в отдельном терминале (`uvicorn app.main:app --reload`), затем выполните ячейку ниже.

Используем `httpx` — HTTP-клиент, который имитирует браузер.

In [ ]:
import httpx

BASE = "http://127.0.0.1:8000"

with httpx.Client(base_url=BASE, follow_redirects=False) as client:
    # Регистрация
    r = client.post("/register", data={"email": "notebook@test.com", "password": "test1234"})
    print("Register:", r.status_code, "->", r.headers.get("location"))
    cookies = r.cookies

    # Дашборд
    r = client.get("/", cookies=cookies)
    print("Dashboard:", r.status_code, "(logged in:", "notebook@test.com" in r.text, ")")

    # Публикация отчёта
    sample = project_root / "sample_report.pdf"
    with sample.open("rb") as f:
        r = client.post(
            "/reports/publish",
            data={"title": "Test Avito Report"},
            files={"file": ("sample_report.pdf", f, "application/pdf")},
            cookies=cookies,
        )
    print("Publish:", r.status_code, "->", r.headers.get("location"))

    r = client.get("/", cookies=cookies)
    print("Has report on page:", "Test Avito Report" in r.text)

print("\nГотово! Откройте http://127.0.0.1:8000 в браузере для ручной проверки.")

## Часть 7 — Что дальше

Вы прошли полный цикл **сайт + БД**:

1. Спроектировали таблицы
2. Подключили ORM (SQLAlchemy)
3. Добавили авторизацию (хеш пароля + сессия)
4. Реализовали бизнес-логику (⭐)
5. Связали всё через FastAPI и HTML

### Следующие шаги для реального проекта

- **PostgreSQL** вместо SQLite (тот же SQLAlchemy, другой URL подключения)
- **Отправка email** (SMTP / SendGrid) для подтверждения регистрации
- **JWT-токены** вместо cookie-сессии для мобильного API
- **React/Vue** фронтенд + FastAPI только как JSON API
- **Docker + деплой** на VPS или Railway/Render
- **Миграции** через Alembic (версионирование схемы БД)

### Структура проекта

```
autoteka_stars/
├── app/           ← Python-код
├── templates/     ← HTML-шаблоны
├── uploads/       ← файлы отчётов
├── autoteka.db    ← база SQLite
└── web_db_tutorial.ipynb  ← этот блокнот
```